[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzobasile/DeepLearningMHPC/blob/main/2a_unsupervised.ipynb)

# Lab 2: Learning without labels - PCA, K-means and Autoencoders

## Before you start: clone the repository

This notebook is part of the course repository and uses files from it (e.g. the figures in the `notebooks_imgs/` folder). **Do not download this notebook on its own**: clone the whole repository first and open the notebook from inside the cloned folder, otherwise the paths to those files will not work.

**Locally** (from a terminal):

```bash
git clone <REPO_URL> mdmc_dl_hands_on
cd mdmc_dl_hands_on
jupyter lab
```

**On Google Colab**, run this in a code cell before anything else:

```python
!git clone <REPO_URL> mdmc_dl_hands_on
%cd mdmc_dl_hands_on
```


Every lab will finish with a small homework that builds on the day's material. We can briefly discuss any doubt or curiosity the following day. The notebooks with the solutions will be published after the whole course has ended.

## How to use this notebook

This is the **student version** of the lab: some code cells are incomplete, and completing them is your job.

- **🛠 Exercise** cells (in markdown) explain what to implement, with hints on which functions to use.
- The code cell right after each exercise contains `# TODO` comments and `...` placeholders: **replace every `...` with your code**. Everything else in the cell is already written for you: read it, you will need it!
- Some exercises are followed by a **✅ Check** cell: run it after your solution. If it prints `All good!` (or the expected values), you can move on.
- Run the cells **in order**: later exercises use variables and functions defined in earlier ones.
- If you get stuck: read the docstrings and comments, the explanation cells above the exercise and the linked documentation, or ask!

## What we'll do in this second hands on session ?

In Lab 1 every image came with its **label** (*"this is a 3"*), and we trained MLPs and CNNs to predict it. Today we throw the labels away and ask: **what can a model learn from the images alone?**

The objective of this hands on session is:
- understand what **unsupervised learning** is and why it is so important in deep learning;
- use two classical unsupervised tools as a starting point: **PCA** (to compress the data) and **K-means** (to find groups in the data);
- build and train our first unsupervised neural network, the **autoencoder**, and compare it with PCA;
- use what the autoencoder learned for three practical tasks: **clustering**, **anomaly detection** and **classification with very few labels**.

We will reuse many tools from Lab 1 (dataset, `DataLoader`, training loop, `plot_curves`), so the new things are the ideas, the models and the losses.

# What is unsupervised learning?

## Supervised vs unsupervised

In **supervised learning** (Lab 1) the dataset is made of pairs $(x, y)$: an input and the answer we want. The model learns the map $x \to y$, and the loss compares the prediction with the true answer.

In **unsupervised learning** the dataset is made of inputs $x$ only. There is no "correct answer" to predict: the goal is to discover the **structure** hidden in the data.

| | Supervised | Unsupervised |
|---|---|---|
| Data | pairs $(x, y)$ | only $x$ |
| Question | *"What is the label of $x$?"* | *"What does the data look like? How is it organised?"* |
| Loss | compares the prediction with the label $y$ | uses only $x$ (e.g. *"can I rebuild $x$?"*) |
| Example (Lab 1 / today) | MLP classifier on MNIST | PCA, K-means, autoencoder on MNIST |

## What can we do without labels?

The most common unsupervised tasks are:

- **Dimensionality reduction:** describe every data point with a few numbers instead of many (an MNIST image has 784 pixels, but we will see that a handful of numbers is enough to describe it). Useful to compress, to visualize and as a first step for other tasks.
- **Clustering:** split the data into groups of similar points (e.g. group customers by behaviour, documents by topic, cells by gene expression).
- **Anomaly detection:** find the points that do not look like the others (fraud, faulty sensors, defective products).
- **Generation:** learn the distribution of the data to create new examples (the models behind image generators and LLMs).

## Why does it matter so much in deep learning?

**Labels are expensive, data is cheap.** Collecting millions of images or texts is easy; asking experts to label all of them is slow and costly (think of medical images). Deep networks need *a lot* of data, so the question becomes: can we learn something useful from the unlabelled data, and then use only a few labels?

This is exactly how modern deep learning works: large models are first **pre-trained** on huge amounts of unlabelled data with a task created *from the data itself* (e.g. *"predict the next word"* for LLMs, *"rebuild the input"* for autoencoders). This is called **self-supervised learning**. Then the model is **fine-tuned** with few labels for the task we care about. Today we will do a small version of this whole pipeline.

## How do we check the results without labels?

This is the hard part of unsupervised learning: there is no accuracy to compute. In this lab we use a trick: MNIST **does** have labels, but we keep them **hidden** during training. We use them only at the end, to **colour the plots** and to **check** whether what the model found makes sense. Remember: *the models never see the labels* (until the last section, where we use only a few of them).

## The plan

1. **PCA:** a linear method to compress the images, and a first look at the data in 2D.
2. **K-means:** a simple algorithm to group the images, and the limits of measuring similarity pixel by pixel.
3. **Autoencoder:** a neural network that learns to compress and rebuild images, a non-linear PCA.
4. **Using the learned representation:** clustering, anomaly detection and classification with few labels.

## Library import

In [ ]:
import copy

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split
from torchvision.transforms import v2
from torchvision.datasets import MNIST, FashionMNIST
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# use the best available device: NVIDIA GPU (cuda) > Apple Silicon GPU (mps) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

_ = torch.manual_seed(0)  # fix the random seed: same initial weights, same shuffling, (almost) same results

Same rule as in Lab 1: **the model and the data must be on the same device** (`model.to(DEVICE)`, `x.to(DEVICE)`), and results that go to NumPy, scikit-learn or matplotlib must be moved back to the CPU first (`.cpu()`).

# Dataset preparation

We use MNIST again, with one difference from Lab 1: **we do not normalize** the images, we only scale them to $[0, 1]$.

| Transform | What it does | MNIST image after the transform |
|---|---|---|
| `v2.ToImage()` | converts the PIL image into a tensor `(channels, height, width)` | `uint8`, shape `(1, 28, 28)`, values in $\{0, \dots, 255\}$ |
| `v2.ToDtype(torch.float32, scale=True)` | converts to float and divides by 255 | `float32`, values in $[0, 1]$ |

**Why no `v2.Normalize` this time?** Today our neural network has to *produce* images, not only read them. It is much easier to compare (and to plot) the output with the input if both are simple images with pixels in $[0, 1]$: 0 is black, 1 is white.

In [ ]:
def plot_images(images: torch.Tensor, titles=None, n_cols: int = 10, suptitle: str = None, size: float = 1.3):
    """Plot a batch of images of shape (batch, 1, 28, 28) (or (batch, 784)) in a grid with `n_cols` columns."""
    # Pixels are shown in gray scale with a fixed range [0, 1]: black = 0, white = 1.
    images = torch.as_tensor(images).detach().cpu().reshape(-1, 28, 28)
    n_rows = -(-len(images) // n_cols)  # ceil division
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows + 0.4), squeeze=False)
    for ax in axes.flat:
        ax.set_axis_off()  # also blanks any unused cells in the last row
    for i, (ax, img) in enumerate(zip(axes.flat, images)):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        if titles is not None:
            ax.set_title(titles[i], fontsize=8)
    if suptitle is not None:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

In [ ]:
transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])
mnist_train = MNIST('datasets', train=True, download=True, transform=transforms)
mnist_test = MNIST('datasets', train=False, download=True, transform=transforms)

X, y = zip(*(mnist_train[i] for i in range(10)))
X = torch.stack(X)
print("shape:", X.shape, "- min:", X.min().item(), "- max:", X.max().item())
plot_images(X, titles=list(y))

### DataLoader

As in Lab 1 we keep a part of the training set as **validation set** to monitor the training, and we use the test set only at the end. Today we do not tune hyperparameters with cross-validation, so a simple random split is enough: `torch.utils.data.random_split` does it in pure PyTorch (same 48k / 12k split we used in Lab 1 with `ValidSplit`).

We use a larger batch size than in Lab 1 (128 instead of 32): the models of today are small and training is faster.

**Note:** the `DataLoader`s still return pairs `(images, labels)`, because the dataset has labels. In the unsupervised part we will simply **ignore** the labels.

### 🛠 Exercise 1 — Split and DataLoaders

1. Split `mnist_train` into 48k training and 12k validation images with `random_split(dataset, [48000, 12000], generator=...)`. Pass `generator=torch.Generator().manual_seed(0)`, so that the split is the same at every run.
2. Create the three `DataLoader`s with `batch_size=BATCH_SIZE`. As in Lab 1, shuffle only the training data.

In [ ]:
# TODO: 48k images for training, 12k for validation; the generator makes the split reproducible
mnist_train_split, mnist_val = ...

BATCH_SIZE = 128
# TODO: the three DataLoaders (shuffle only the training one)
mnist_train_loader = ...
mnist_val_loader = ...
mnist_test_loader = ...

print(f"train: {len(mnist_train_split)} - val: {len(mnist_val)} - test: {len(mnist_test)}")

### The data as a table

PCA and K-means come from **scikit-learn**, which works with NumPy arrays of shape `(n_samples, n_features)`: one row per image, one column per pixel. So we also build the "flat" version of our images, `(n_images, 784)`.

The labels `y` are returned too, but remember: they are used **only** to check the results and to colour the plots.

### 🛠 Exercise 2 — From images to a table

Complete `get_arrays`: every mini-batch `x` has shape `(batch, 1, 28, 28)`. Flatten it into `(batch, 784)`, one row per image, keeping the batch dimension (same trick as the `forward` of the MLP in Lab 1).

In [ ]:
def get_arrays(dataloader):
    """Run through `dataloader` and return all the images as a (n_images, 784) array and the labels as a (n_images,) array."""
    X_all, y_all = [], []
    for x, y in dataloader:
        X_all.append(...)   # TODO: (batch, 1, 28, 28) -> (batch, 784)
        y_all.append(y)
    return torch.cat(X_all).numpy(), torch.cat(y_all).numpy()


# a DataLoader WITHOUT shuffling, so that the order of the images is always the same
X_train, y_train = get_arrays(DataLoader(mnist_train_split, batch_size=1000))   # 48k training images
X_test, y_test = get_arrays(mnist_test_loader)      # 10k test images
print("X_train:", X_train.shape, "- X_test:", X_test.shape)   # expected: (48000, 784) and (10000, 784)

# Part 1: Dimensionality reduction with PCA

## The idea

An MNIST image is a point in a space with **784 dimensions**, one per pixel. But if we chose 784 random numbers we would get noise, never a digit: real digits occupy only a tiny, well organised part of this space. Neighbouring pixels are strongly correlated (a stroke is made of many white pixels next to each other) and a digit can be described by a few **factors**: which digit it is, how thick, how slanted, how wide, ...

**Principal Component Analysis (PCA)** looks for the few directions of the 784-dimensional space along which the images change the most (the directions of **maximum variance**), called **principal components**.

**Intuition.** Think of the shadow of an object: it is a 2D description of a 3D object. If we choose the direction of the light well, the shadow keeps most of the information about the shape (the shadow of a hand can look like a rabbit!). PCA chooses the "light" that gives the most informative shadow of our data.

## PCA as an encoder and a decoder

With $k$ principal components $w_1, \dots, w_k$ (each one is a vector of 784 numbers, i.e. an image!) and the mean image $\bar{x}$:

- **encode** (compress): every image $x$ becomes $k$ numbers, its coordinates along the components
$$z_i = w_i^T (x - \bar{x}), \qquad i = 1, \dots, k$$
- **decode** (rebuild): the image is approximated as the mean image plus a mix of the components
$$\hat{x} = \bar{x} + z_1 w_1 + z_2 w_2 + \dots + z_k w_k$$

PCA chooses the components that make the **reconstruction error** $\|x - \hat{x}\|^2$ as small as possible (on average over the dataset). Keep in mind the words *encode*, *decode*, *reconstruction error*: they will come back with the autoencoder.

In scikit-learn:

| Method | What it does | Shapes |
|---|---|---|
| `PCA(n_components=k).fit(X)` | computes the mean image and the $k$ components on the data `X` | `X`: `(n, 784)` |
| `pca.transform(X)` | encodes: $x \to z$ | `(n, 784)` → `(n, k)` |
| `pca.inverse_transform(Z)` | decodes: $z \to \hat{x}$ | `(n, k)` → `(n, 784)` |

Note that `fit` receives only `X`: **no labels**.

### 🛠 Exercise 3 — PCA in 2D

Use the scikit-learn interface of the table above:
1. create a `PCA` with 2 components and `fit` it on the training images `X_train`. Only the images: **no labels**! (`fit` returns the fitted object itself, so you can write it in one line.)
2. encode the test images `X_test` with `transform`: you get one 2D point (two numbers) per image.

`plot_latent_space` (given) draws the points, coloured by the true digit.

In [ ]:
def plot_latent_space(z, y, title, ax=None):
    """Scatter plot of 2D codes z (one point per image), coloured by the labels y."""
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(z[:, 0], z[:, 1], c=y, cmap="tab10", s=2, alpha=0.7, vmin=-0.5, vmax=9.5)
    cbar = plt.colorbar(sc, ax=ax, ticks=range(10))
    cbar.set_label("digit (used only for the colours)")
    ax.set_title(title)
    ax.set_xlabel("$z_1$")
    ax.set_ylabel("$z_2$")
    if show:
        plt.show()


pca_2 = ...     # TODO: PCA with 2 components, fitted on the training images
pca_2_z = ...   # TODO: encode the test images: (10000, 784) -> (10000, 2)
print("codes shape:", pca_2_z.shape)

plot_latent_space(pca_2_z, y_test, title="PCA - the test set in 2D")

Some digits are already well separated (look at the 0s and the 1s: the 1s have very few white pixels, the 0s very many), but most of them are mixed in the middle. Two numbers obtained with a **linear** projection are not enough to separate ten digits.

## How many components do we need?

Every component explains a part of the total variance of the data (`pca.explained_variance_ratio_`). Let's fit PCA with all the 784 components and look at how much variance we keep with the first $k$.

### 🛠 Exercise 4 — Explained variance

1. Fit a `PCA` **without** specifying `n_components`: it keeps all the 784 components.
2. `pca_full.explained_variance_ratio_` contains, for every component, the fraction of the variance it explains. Compute the **cumulative** sum with `np.cumsum`: element `k-1` is the variance explained by the first `k` components together.

The plot is given.

In [ ]:
pca_full = ...     # TODO: PCA with all the 784 components, fitted on X_train
cumulative = ...   # TODO: variance explained by the first k components, for every k

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(cumulative) + 1), cumulative)
for target in [0.5, 0.8, 0.9, 0.95]:
    k = int(np.searchsorted(cumulative, target)) + 1          # first k that reaches the target
    ax.axhline(target, color="gray", linestyle="--", linewidth=0.8)
    ax.plot(k, cumulative[k - 1], "o", color="black")
    ax.annotate(f"{target:.0%} with k={k}", (k, cumulative[k - 1]), xytext=(10, -12), textcoords="offset points")
ax.set_xscale("log")
ax.set_title("PCA - explained variance")
ax.set_xlabel("number of components k (log scale)")
ax.set_ylabel("fraction of variance explained")
ax.grid(alpha=0.3)
plt.show()

About 90% of the variance is kept with fewer than 100 components out of 784: MNIST is highly **compressible**.

## What do the components look like?

Each component is a vector of 784 numbers, so we can draw it as an image. Red pixels are positive, blue pixels are negative: moving along a component adds the red pixels and removes the blue ones from the mean image.

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(pca_full.mean_.reshape(28, 28), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("mean image")
for i, ax in enumerate(axes[1:]):
    component = pca_full.components_[i].reshape(28, 28)
    limit = np.abs(component).max()
    ax.imshow(component, cmap="RdBu_r", vmin=-limit, vmax=limit)   # red = positive, blue = negative
    ax.set_title(f"component {i + 1}")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

The first components are smooth "blobs" that look like parts of digits (a 0 against a 1, a slanted stroke, ...).

## Reconstructions

Let's encode and decode some test images keeping $k$ components, and measure the **reconstruction error** as the mean squared error per pixel (MSE):
$$\text{MSE} = \frac{1}{784} \sum_{j=1}^{784} (x_j - \hat{x}_j)^2$$

### 🛠 Exercise 5 — Reconstructions and reconstruction error

1. Complete `mse_per_image`: given two arrays of shape `(n, 784)`, return for every image the mean over its pixels of the squared difference (the formula above), i.e. an array of shape `(n,)`. *Hint:* `.mean(axis=...)`: along which axis are the pixels?
2. In the loop, **encode and then decode** the test images with `pca_k` (`transform`, then `inverse_transform`) to get the reconstructions `X_rec`, of shape `(10000, 784)`.

Expected: the error decreases as `k` grows, from about `0.056` with 2 components to about `0.003` with 150.

In [ ]:
def mse_per_image(X, X_rec):
    """Mean squared error per pixel of every image: arrays of shape (n, 784) -> (n,)."""
    return ...  # TODO


examples = X_test[:10]
rows, titles = [examples], ["original"] + [""] * 9
for k in [2, 10, 50, 150]:
    pca_k = PCA(n_components=k).fit(X_train)
    X_rec = ...   # TODO: encode and decode the test images with pca_k
    print(f"k = {k:>3} components - test MSE per pixel: {mse_per_image(X_test, X_rec).mean():.4f}")
    rows.append(np.clip(X_rec[:10], 0, 1))   # the reconstruction can go slightly outside [0, 1]
    titles += [f"k = {k}"] + [""] * 9

plot_images(np.concatenate(rows), titles=titles, n_cols=10, suptitle="PCA - top: original, then reconstructions with k components")

In [ ]:
# ✅ Check
assert np.allclose(mse_per_image(np.zeros((3, 784)), np.full((3, 784), 0.5)), [0.25, 0.25, 0.25]), "mse_per_image is not correct"
print("All good!")

With 2 components the reconstructions are blurry blobs, with 50 they are already good, with 150 almost perfect.

**Take-home message:** PCA compresses the data **without labels**, by finding the directions where the data changes the most. Its limit: it is **linear**, i.e. it can only rebuild an image as a weighted sum of fixed images. Keep it in mind for later.

# Part 2: Clustering with K-means

## The idea

**Clustering** means splitting the data into groups (**clusters**) so that points in the same group are similar and points in different groups are different. *Similar* means **close**: here we use the usual Euclidean distance.

**K-means** is the simplest clustering algorithm. We choose the number of clusters $K$, and every cluster is represented by its centre $c_k$ (the **centroid**). Then:

```
Choose K random points as initial centroids
Repeat until the centroids stop moving:
    1. Assignment: assign every point to its closest centroid
    2. Update:     move every centroid to the mean of the points assigned to it
```

Every step makes the total distance between the points and their centroid smaller, so the algorithm always stops. It can stop in a bad solution, so in practice it is run several times from different initial centroids (parameter `n_init` of scikit-learn), keeping the best result.

Let's see it in action on a 2D toy dataset: the whole algorithm is just the two steps in `kmeans_step`. We start from a bad initialization on purpose, three points that are very close to each other, and we repeat the two steps until the centroids stop moving.

### 🛠 Exercise 6 — K-means by hand

Complete `kmeans_step`, which performs **one** iteration of the algorithm above. `points` has shape `(n_points, 2)` and `centroids` has shape `(K, 2)`.

1. **Distances.** Compute `distances`, of shape `(n_points, K)`, where `distances[i, k]` is the Euclidean distance between point `i` and centroid `k`.
   *Hint (broadcasting):* `points[:, None, :]` has shape `(n_points, 1, 2)` and `centroids[None, :, :]` has shape `(1, K, 2)`: their difference has shape `(n_points, K, 2)`, all the point–centroid differences at once. Then take the norm along the last axis with `np.linalg.norm(..., axis=2)`.
2. **Assignment.** For every point, the index of the closest centroid: `argmin` along the axis of the centroids.
3. **Update.** Every new centroid is the mean of the points assigned to it: `points[assignment == k]` selects the points of cluster `k`, and `.mean(axis=0)` averages them. Do it for every `k` and stack the `K` results into an array of shape `(K, 2)` with `np.stack([...])`.

The loop that repeats the step until the centroids stop moving and the plots are given.

In [ ]:
from sklearn.datasets import make_blobs

# toy dataset: 300 points in 2D, made of 3 groups (we ignore the true group of each point)
points, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=3)


def kmeans_step(points, centroids):
    """One iteration of K-means: assign every point to its closest centroid, then move the centroids."""
    # TODO 1. assignment: distances between every point and every centroid, shape (n_points, K)
    distances = ...
    assignment = ...
    # TODO 2. update: every centroid becomes the mean of its points, shape (K, 2)
    new_centroids = ...
    return assignment, new_centroids


centroids = points[[5, 6, 7]]  # bad initialization on purpose: 3 points close to each other
history = []                     # (assignment, centroids) at every iteration, to plot them
for it in range(100):
    assignment, new_centroids = kmeans_step(points, centroids)
    history.append((assignment, centroids))
    if np.allclose(new_centroids, centroids):   # the centroids stopped moving: done
        break
    centroids = new_centroids
print(f"K-means converged after {len(history)} iterations")

to_plot = [0, 1, 2, len(history) - 1]   # the first 3 iterations and the last one
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, it in zip(axes, to_plot):
    assignment, centroids_it = history[it]
    ax.scatter(points[:, 0], points[:, 1], c=assignment, cmap="tab10", vmin=0, vmax=9, s=10)
    ax.scatter(centroids_it[:, 0], centroids_it[:, 1], c="black", marker="X", s=150, label="centroids")
    ax.set_title(f"iteration {it + 1}")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Check on a tiny example: two obvious groups
pts = np.array([[0., 0.], [0., 1.], [10., 10.], [10., 11.]])
a, c = kmeans_step(pts, np.array([[0., 0.], [10., 10.]]))
assert list(a) == [0, 0, 1, 1], f"wrong assignment: {a}"
assert np.allclose(c, [[0., 0.5], [10., 10.5]]), f"wrong centroids: {c}"
print("All good!")

Even from a bad start, after a few iterations the centroids settle in the middle of the three groups. (With other data or other initializations K-means can get stuck in a worse solution: this is why scikit-learn runs it `n_init` times.)

## K-means on MNIST

Now we cluster the 48k training images into $K = 10$ groups, using the 784 pixels as coordinates. We choose $K = 10$ because we know there are 10 digits, but **K-means does not know what a digit is**: it only groups images whose pixels are close.

In scikit-learn the interface is the same as PCA: `fit` on the data (no labels), then `predict` gives the cluster of new points.

**Note:** this takes some seconds: every iteration computes the distance between 48k images and 10 centroids in 784 dimensions.

### 🛠 Exercise 7 — K-means on MNIST

1. Create `KMeans(n_clusters=10, n_init=5, random_state=0)` and `fit` it on the training images `X_train` (no labels!).
2. Use `predict` to get the cluster of every **test** image.

In [ ]:
kmeans_pixels = ...     # TODO: K-means with 10 clusters, fitted on the training images
clusters_pixels = ...   # TODO: cluster id of every test image

print("cluster of the first 20 test images:", clusters_pixels[:20])
print("true digit of the first 20 images:  ", y_test[:20])

The cluster ids are just names, from 0 to 9: cluster 3 does **not** mean digit 3. To understand what each cluster contains we can look at the centroids: they are the mean image of each cluster.

In [ ]:
plot_images(kmeans_pixels.cluster_centers_, titles=[f"cluster {k}" for k in range(10)],
            suptitle="K-means on pixels - the 10 centroids")

## Evaluating the clusters

Since we have the (hidden) labels, we can check how well the clusters match the digits.

- **Contingency matrix:** like the confusion matrix of Lab 1, but the columns are clusters, not predicted digits. The cell in row $i$, column $k$ counts the images of digit $i$ that ended up in cluster $k$. A perfect clustering has exactly one large cell in every row and every column.
- **Clustering accuracy:** we give each cluster the name of its most common digit (a *majority vote*) and we compute the accuracy as in Lab 1. A random assignment would give about 10%.

(There are also standard scores that do not need to name the clusters, like the *Adjusted Rand Index* or the *Normalized Mutual Information*, in `sklearn.metrics`. Here we keep the accuracy, which is easier to read.)

### 🛠 Exercise 8 — Clustering accuracy

Complete `cluster_accuracy`. For every cluster `k` (the loop is given):
1. select the true labels of the images in cluster `k`: `y_true[clusters == k]`;
2. count how many images of each digit there are with `np.bincount(...)`. The **largest** count is the number of images that are "correct" if we name cluster `k` after its most common digit;
3. add it to `correct`.

`plot_contingency` is given. Expected: a clustering accuracy of about `0.58`.

In [ ]:
from sklearn.metrics import confusion_matrix


def cluster_accuracy(y_true, clusters):
    """Name each cluster after its most common label, then compute the accuracy."""
    correct = 0
    for k in np.unique(clusters):
        labels_in_cluster = ...   # TODO: true labels of the images in cluster k
        correct += ...            # TODO: images of the most common digit of cluster k
    return correct / len(y_true)


def plot_contingency(y_true, clusters, title):
    """Rows: true digits, columns: clusters. Each cell counts the images."""
    cm = confusion_matrix(y_true, clusters)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for k in range(cm.shape[1]):
            ax.text(k, i, cm[i, k], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, k] > cm.max() / 2 else "black")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_xlabel("cluster (found by K-means)")
    ax.set_ylabel("true digit (hidden during training)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    plt.show()


# ✅ Check on a tiny example: cluster 5 contains digits [0, 0, 1], cluster 7 contains [1] -> (2 + 1) / 4
assert cluster_accuracy(np.array([0, 0, 1, 1]), np.array([5, 5, 5, 7])) == 0.75, "cluster_accuracy is not correct"

acc_pixels = cluster_accuracy(y_test, clusters_pixels)
print(f"K-means on pixels - clustering accuracy on the test set: {acc_pixels:.3f}")
plot_contingency(y_test, clusters_pixels, title="K-means on pixels - contingency matrix (test set)")

Not bad for an algorithm that has never seen a label: some digits (e.g. 2, 3 and 6) end up almost entirely in one cluster. But other clusters mix different digits: typically 4, 7 and 9 share the same clusters, and 5 is mixed with 3 and 8. And since we asked for exactly 10 clusters, when some digits share a cluster other digits must be split into two: typically the straight and the slanted 1s (look at the centroids above).

## The problem: distances between pixels

K-means can find groups without labels, but the result depends on how we **represent** the data. Pixels are a poor representation: we need a representation where *similar digits are close*. We could design it by hand... or **learn** it with a neural network. This is the job of the autoencoder.

# Part 3: Autoencoders

## From PCA to a neural network

PCA has an encoder ($x \to z$) and a decoder ($z \to \hat{x}$), but both are **linear**. An **autoencoder** is the same idea where both are **neural networks**, so they can learn non-linear (curved) relations:

- the **encoder** $f_\phi$ compresses the image into a small vector, called **code** or **latent vector**: $z = f_\phi(x)$, with $\dim(z) \ll 784$;
- the **decoder** $g_\theta$ rebuilds the image from the code: $\hat{x} = g_\theta(z)$.

<p align="center">
  <img src="notebooks_imgs/aes.png" alt="drawing" width="500" style="background-color: white;"/>
</p>

The network is trained to **output its own input**: $\hat{x} \approx x$. This looks like a trivial task, but the middle layer, the **bottleneck**, is very small (today 2 or 10 numbers instead of 784): the network cannot just copy the pixels. To rebuild the images well, the encoder is forced to keep only the most important information, and the decoder to learn how digits are made.

**No labels are used:** the target is the input itself. A task like this, where the "label" is created from the data, is called **self-supervised**.

## The building blocks

| | Layers | Output shape |
|---|---|---|
| Input | flatten | `(batch, 784)` |
| Encoder | `Linear(784, 512)` + ReLU → `Linear(512, 256)` + ReLU → `Linear(256, latent_dim)` | `(batch, latent_dim)` = $z$ |
| Decoder | `Linear(latent_dim, 256)` + ReLU → `Linear(256, 512)` + ReLU → `Linear(512, 784)` + **Sigmoid** | `(batch, 784)` |
| Output | reshape | `(batch, 1, 28, 28)` = $\hat{x}$ |

The decoder is the mirror of the encoder. Notes:
- these are the same `Linear` + ReLU layers of the MLP of Lab 1, nothing new;
- the last layer of the encoder has **no activation**: the code $z$ can be any real number;
- the last layer of the decoder has a **sigmoid**, $\sigma(t) = 1 / (1 + e^{-t})$, that squeezes any number into $(0, 1)$: the output pixels are in the same range as the input pixels.

### `torch.nn.Sequential`

In Lab 1 we defined every layer as an attribute and we called them one by one in `forward`. When the layers are simply applied one after the other, `torch.nn.Sequential(layer1, layer2, ...)` does it for us: it is a module that applies its layers in order. This makes the encoder and the decoder two separate modules that we can also use on their own (`model.encode(x)`, `model.decode(z)`).

### 🛠 Exercise 9 — The autoencoder

Complete the `AutoEncoder` following the table above.

- `self.encoder`: a `torch.nn.Sequential` with `Linear(784, 512)`, ReLU, `Linear(512, 256)`, ReLU, `Linear(256, latent_dim)`. No activation at the end.
- `self.decoder`: the mirror: `Linear(latent_dim, 256)`, ReLU, `Linear(256, 512)`, ReLU, `Linear(512, 784)`, and a `torch.nn.Sigmoid()` at the end (output pixels in $[0, 1]$).
- `encode(x)`: flatten the images to `(batch, 784)` and apply the encoder.
- `decode(z)`: apply the decoder and reshape its output into images `(batch, 1, 28, 28)` (use `-1` for the batch dimension).
- `forward(x)`: encode, then decode.

The next cell checks the shapes: `torch.Size([128, 2])` for the code and `torch.Size([128, 1, 28, 28])` for the output.

In [ ]:
class AutoEncoder(torch.nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        # TODO: encoder: (batch, 784) -> (batch, latent_dim)
        self.encoder = torch.nn.Sequential(
            ...
        )
        # TODO: decoder: (batch, latent_dim) -> (batch, 784), the mirror of the encoder, pixels in [0, 1]
        self.decoder = torch.nn.Sequential(
            ...
        )

    def encode(self, x):
        return ...   # TODO: (batch, 1, 28, 28) -> (batch, latent_dim)

    def decode(self, z):
        return ...   # TODO: (batch, latent_dim) -> (batch, 1, 28, 28)

    def forward(self, x):
        z = ...      # TODO
        x_hat = ...  # TODO
        return x_hat


ae_2 = AutoEncoder(latent_dim=2).to(DEVICE)   # a bottleneck of 2 numbers, like PCA with k=2
print(ae_2)

In [ ]:
# check the shapes
fake_input = torch.rand(BATCH_SIZE, 1, 28, 28, device=DEVICE)
print("code size:  ", ae_2.encode(fake_input).size())
print("output size:", ae_2(fake_input).size())

## The loss: reconstruction error

We want $\hat{x}$ to be as close as possible to $x$, so we use the same reconstruction error of PCA, the **mean squared error** over the pixels (`torch.nn.functional.mse_loss`):
$$\mathcal{L}(x, \hat{x}) = \frac{1}{784} \sum_{j=1}^{784} (x_j - \hat{x}_j)^2$$
averaged, as usual, over the images of the mini-batch. Using the same error as PCA makes the comparison between the two methods fair.

Compare it with the cross-entropy of Lab 1: there the loss compared the **output with the label** $y$; here it compares the **output with the input** $x$. That is the whole difference between our supervised and unsupervised training.

### A training loop without labels

`train_one_epoch` of Lab 1 needs the labels `y`. For the autoencoder we write a version where the labels are **ignored** (`for x, _ in dataloader`) and the loss compares the output with `x`. Everything else (forward, `zero_grad`, `backward`, `step`) is exactly as in Lab 1.

### 🛠 Exercise 10 — Training without labels

Complete the two functions, starting from `train_one_epoch` and `evaluate` of Lab 1. The differences:
- the labels are ignored: the loop is `for x, _ in dataloader`, and only `x` goes to `DEVICE`;
- the loss is `F.mse_loss(x_hat, x)`: the target is **the input itself**;
- `evaluate_unsupervised` returns only the average loss (there is no accuracy). As in Lab 1, `F.mse_loss` returns the mean over the batch: multiply it by `len(x)` before adding it to the total.

Finally compute the sanity check `lazy_mse`: the test error of a "lazy" model that ignores the input and always outputs the mean training image. The mean image is `X_train.mean(axis=0, keepdims=True)` (shape `(1, 784)`): pass it to `mse_per_image` together with `X_test` (broadcasting compares it with every test image) and average the result.

Expected: about `0.23` for the untrained AE and about `0.068` for the lazy model.

In [ ]:
def train_one_epoch_unsupervised(model, dataloader, optimizer):
    """Train the autoencoder for one epoch WITHOUT the labels and return the average loss."""
    model.train()
    epoch_loss = 0
    for x, _ in dataloader:          # the labels are not used!
        x = x.to(DEVICE)
        x_hat = ...                  # TODO: forward pass
        l = ...                      # TODO: the target is the input itself
        ...                          # TODO: erase the past gradients
        ...                          # TODO: backward pass
        ...                          # TODO: update the parameters
        epoch_loss += l.item()
    return epoch_loss / len(dataloader)


def evaluate_unsupervised(model, dataloader):
    """Average reconstruction loss (MSE per pixel) of the autoencoder on `dataloader`."""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(DEVICE)
            total_loss += ...        # TODO: MSE of the batch (a Python number) times the batch size
    return total_loss / len(dataloader.dataset)


# TODO: sanity check: a "lazy" model that ignores the input and always outputs the mean image
lazy_mse = ...
print(f"untrained AE - val loss: {evaluate_unsupervised(ae_2, mnist_val_loader):.4f}")
print(f"always the mean image - test loss: {lazy_mse:.4f}  <- the trained AE must do much better than this")

## Training the autoencoder

We use `torch.optim.Adam` with its default learning rate `lr=1e-3`: as we saw in Lab 1, it converges fast without much tuning.

### 🛠 Exercise 11 — The training function

`plot_curves` is the same as in Lab 1. Complete `train_autoencoder`:
1. create an Adam optimizer (`torch.optim.Adam`) on the parameters of `model`, with `lr=1e-3`;
2. at every epoch, train on `mnist_train_loader` with `train_one_epoch_unsupervised` and compute the validation loss on `mnist_val_loader` with `evaluate_unsupervised`.

In [ ]:
def plot_curves(curves, title, ylabel, reference=None):
    """Same as Lab 1: plot one or more lists of numbers (one value per epoch)."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, values in curves.items():
        epochs_axis = range(1, len(values) + 1)  # epochs start from 1
        ax.plot(epochs_axis, values, marker="o", label=label)
    if reference is not None:
        ref_label, ref_value = reference
        ax.axhline(ref_value, color="gray", linestyle="--", label=ref_label)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


def train_autoencoder(model, epochs: int):
    """Train `model` with Adam for `epochs` epochs and return the training and validation losses."""
    optimizer = ...   # TODO: Adam with lr=1e-3
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        train_loss = ...   # TODO: one epoch of training on the training set
        val_loss = ...     # TODO: reconstruction loss on the validation set
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f} - val loss: {val_loss:.4f}")
    return train_losses, val_losses

### 🛠 Exercise 12 — Autoencoder vs PCA

1. Train `ae_2` for 10 epochs with `train_autoencoder`.
2. Compute its test loss with `evaluate_unsupervised` on `mnist_test_loader`.
3. Compute the test loss of PCA with 2 components, for comparison: decode the codes `pca_2_z` of Exercise 3 with `pca_2.inverse_transform`, compute `mse_per_image` with `X_test` and average.

Expected: about `0.038` for the AE and `0.056` for PCA.

In [ ]:
ae_2_train_losses, ae_2_val_losses = ...   # TODO: train ae_2 for 10 epochs

ae_2_test_loss = ...    # TODO: test reconstruction loss of the AE
pca_2_test_loss = ...   # TODO: test reconstruction loss of PCA with 2 components
print(f"AE  (2 numbers) - test MSE per pixel: {ae_2_test_loss:.4f}")
print(f"PCA (2 numbers) - test MSE per pixel: {pca_2_test_loss:.4f}")

plot_curves({"train": ae_2_train_losses, "validation": ae_2_val_losses},
            title="Autoencoder (latent_dim=2) - loss", ylabel="MSE per pixel",
            reference=("PCA with 2 components (test)", pca_2_test_loss))

With the same bottleneck of 2 numbers, the autoencoder rebuilds the images much better than PCA: this is the advantage of a **non-linear** encoder and decoder.

### Reconstructions

Let's look at it with our eyes: the same test images, compressed to 2 numbers by PCA and by the AE.

In [ ]:
def ae_reconstruct(model, X):
    """Encode and decode the (n, 784) array X with the autoencoder; return a (n, 784) array."""
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(X).to(DEVICE)).cpu().reshape(len(X), -1).numpy()


examples = X_test[:10]
plot_images(np.concatenate([examples,
                            np.clip(pca_2.inverse_transform(pca_2.transform(examples)), 0, 1),
                            ae_reconstruct(ae_2, examples)]),
            n_cols=10, suptitle="top: original - middle: PCA (2 numbers) - bottom: autoencoder (2 numbers)")

The AE reconstructions are still blurry (2 numbers are really few!) but most digits are recognisable, while PCA gives shapeless blobs.

## The latent space

Every image is now a point $z \in \mathbb{R}^2$, so we can draw the test set as we did with PCA. Again, **the labels are used only for the colours, the autoencoder has never seen them**.

### 🛠 Exercise 13 — Getting the codes

Complete `get_codes`, which returns the codes $z$ of the images. It is very similar to `ae_reconstruct` (in the cell above): the difference is that it uses only the **encoder** (`model.encode`), not the whole model. Remember to go from NumPy to a tensor on `DEVICE`, and back to NumPy on the CPU at the end.

In [ ]:
def get_codes(model, X):
    """Encode the (n, 784) array X with the encoder of `model`; return a (n, latent_dim) array."""
    model.eval()
    with torch.no_grad():
        return ...   # TODO


ae_2_z = get_codes(ae_2, X_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_latent_space(pca_2_z, y_test, title="PCA - 2D codes of the test set", ax=axes[0])
plot_latent_space(ae_2_z, y_test, title="Autoencoder - 2D codes of the test set", ax=axes[1])
plt.show()

This is the most important plot of the lab. **Without any label**, the autoencoder has organised the images so that the same digits are close to each other: to rebuild the images well, it had to discover that there are different "kinds" of images. Similar digits are neighbours (e.g. 4 and 9, or 3, 5 and 8), the same pairs that the classifier of Lab 1 confused most often.

Compare the grid with the scatter plot: each region of the plane decodes into the digit that lives there, and moving in the plane changes the digit, its slant or its thickness.

**Can we use this to generate new digits?** Partially: in the empty regions between the clusters the decoder was never trained, and it often produces strange shapes. Moreover, we do not know where to pick $z$ to get a "typical" digit. Fixing these problems leads to **generative models** such as the **Variational Autoencoder (VAE)**: see the summary at the end.

## A larger bottleneck

2 numbers are great to draw the latent space, but too few to describe a digit well. Let's train the same autoencoder with a code of **10 numbers** and compare it with PCA with 10 components. (10 is not related to the number of digits: try other values in the homework.)

### 🛠 Exercise 14 — A larger bottleneck

1. Create an `AutoEncoder` with `latent_dim=LATENT_DIM` (on `DEVICE`) and train it for 10 epochs.
2. Fit a `PCA` with `LATENT_DIM` components on `X_train`, for the comparison.

In [ ]:
LATENT_DIM = 10

ae_10 = ...                                     # TODO
ae_10_train_losses, ae_10_val_losses = ...      # TODO: train it for 10 epochs

pca_10 = ...                                    # TODO

plot_curves({"train": ae_10_train_losses, "validation": ae_10_val_losses},
            title=f"Autoencoder (latent_dim={LATENT_DIM}) - loss", ylabel="MSE per pixel",
            reference=(f"PCA with {LATENT_DIM} components (test)",
                       mse_per_image(X_test, pca_10.inverse_transform(pca_10.transform(X_test))).mean()))

In [ ]:
results = {
    "always the mean image": lazy_mse,
    "PCA, 2 components": pca_2_test_loss,
    "AE, latent_dim=2": ae_2_test_loss,
    f"PCA, {LATENT_DIM} components": mse_per_image(X_test, pca_10.inverse_transform(pca_10.transform(X_test))).mean(),
    f"AE, latent_dim={LATENT_DIM}": evaluate_unsupervised(ae_10, mnist_test_loader),
}
for name, value in results.items():
    print(f"{name:<24} - test MSE per pixel: {value:.4f}")

plot_images(np.concatenate([examples,
                            np.clip(pca_10.inverse_transform(pca_10.transform(examples)), 0, 1),
                            ae_reconstruct(ae_10, examples)]),
            n_cols=10, suptitle=f"top: original - middle: PCA ({LATENT_DIM} numbers) - bottom: autoencoder ({LATENT_DIM} numbers)")

With 10 numbers the autoencoder rebuilds the digits quite well, and much better than PCA with the same number of components. Each image of 784 pixels is now summarised by **10 numbers learned without labels**. In the next part we use these 10 numbers as a new **representation** of the images.

# Part 4: Using what the autoencoder learned

Reconstructing images is not very useful in itself: we already have the images! What is useful is what the network learned **on the way**. We can use the trained autoencoder for three tasks (we'll only do point the first one):

1. **Clustering:** K-means on the codes $z$ instead of the pixels;
2. **Anomaly detection:** the reconstruction error tells us whether an input looks like the training data;
3. **Classification with few labels:** reuse the encoder, trained without labels, as the first layers of a classifier.

## 4.1 Clustering in the latent space

We run exactly the same K-means as before, but on the 10-dimensional codes of the autoencoder. As a comparison, we also run it on the 10 PCA components.

### 🛠 Exercise 15 — K-means on the learned codes

1. Encode the training and the test images with `ae_10` (`get_codes`) and with `pca_10` (`transform`).
2. Fit a `KMeans` (same parameters as in Exercise 7) on the **training codes** of the AE, and predict the clusters of the **test codes**. Do the same with the PCA codes.

Expected: the autoencoder codes give a clearly better clustering accuracy (about `0.72`) than the pixels and PCA (about `0.58`).

In [ ]:
codes_train, codes_test = ...   # TODO: AE codes, (48000, 10) and (10000, 10)
pca_train, pca_test = ...       # TODO: PCA codes, (48000, 10) and (10000, 10)

kmeans_ae = ...      # TODO: K-means fitted on the AE training codes
clusters_ae = ...    # TODO: clusters of the AE test codes
clusters_pca = ...   # TODO: the same with the PCA codes (fit on train, predict on test)

print(f"K-means on pixels         (784 numbers) - clustering accuracy: {acc_pixels:.3f}")
print(f"K-means on PCA            ({LATENT_DIM} numbers)  - clustering accuracy: {cluster_accuracy(y_test, clusters_pca):.3f}")
print(f"K-means on autoencoder    ({LATENT_DIM} numbers)  - clustering accuracy: {cluster_accuracy(y_test, clusters_ae):.3f}")

plot_contingency(y_test, clusters_ae, title="K-means on the autoencoder codes - contingency matrix (test set)")

## 4.2 Anomaly detection: what does not look like a digit?

The autoencoder learned to rebuild **digits**. What happens if we give it something else? It will try to rebuild it as a digit, and fail: the **reconstruction error will be large**. So the reconstruction error is a score of *how unusual* an input is, learned without labels and without ever seeing an anomaly.

We compare the reconstruction errors on three sets of images:

- **MNIST test set**: the kind of data the autoencoder was trained on;
- **FashionMNIST test set** (the dataset of the Lab 1 homework): same format, $28 \times 28$ gray images, but clothes instead of digits;
- **random noise**.

### 🛠 Exercise 16 — Reconstruction error as an anomaly score

The three sets of images are loaded for you as `(10000, 784)` arrays. For each of them compute the reconstruction error of **every image** with `ae_10`: reconstruct the images with `ae_reconstruct`, then compare them with the originals using `mse_per_image`. The histogram is given.

In [ ]:
fashion_test = FashionMNIST('datasets', train=False, download=True, transform=transforms)   # same transforms: pixels in [0, 1]
X_fashion, _ = get_arrays(DataLoader(fashion_test, batch_size=BATCH_SIZE))
X_noise = np.random.default_rng(0).random((10000, 784), dtype=np.float32)                 # uniform random pixels

# TODO: reconstruction error of every image, for each set (arrays of shape (10000,))
errors = {
    "MNIST (digits)": ...,
    "FashionMNIST (clothes)": ...,
    "random noise": ...,
}

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.logspace(np.log10(min(e.min() for e in errors.values())), np.log10(max(e.max() for e in errors.values())), 60)
for name, values in errors.items():
    ax.hist(values, bins=bins, alpha=0.6, label=name)
ax.set_xscale("log")
ax.set_title("Autoencoder - reconstruction error of every image")
ax.set_xlabel("MSE per pixel (log scale)")
ax.set_ylabel("number of images")
ax.legend()
plt.show()

The digits have small errors, the clothes much larger ones and the noise is far away. Let's look at some clothes and at how the autoencoder rebuilds them.

In [ ]:
x_fashion_examples = X_fashion[:10]
plot_images(np.concatenate([x_fashion_examples, ae_reconstruct(ae_10, x_fashion_examples)]), n_cols=10,
            titles=[f"error {e:.3f}" for e in errors["FashionMNIST (clothes)"][:10]] + [""] * 10,
            suptitle="FashionMNIST - top: input (and its reconstruction error), bottom: autoencoder reconstruction")

The autoencoder only knows how to draw digits, so it draws everything as a digit: pullovers and shirts become 0s or 8s, trousers become thin 8s!

### A simple anomaly detector

To build a real detector we need a **threshold**: an image is flagged as an anomaly if its error is larger than the threshold. We choose it **using only normal data**: the 99th percentile of the errors on the MNIST validation set (so ~1% of the normal images will be flagged by mistake).

### 🛠 Exercise 17 — A simple anomaly detector

1. Compute the reconstruction errors of the MNIST **validation** images `X_val`, and set `threshold` to their 99th percentile with `np.percentile(values, 99)`.
2. For each set of `errors`, compute the fraction of images flagged as anomalies, i.e. with an error larger than the threshold. *Hint:* `values > threshold` is an array of `True`/`False`, and the mean of a boolean array is the fraction of `True`s.

Expected: about 1% of the digits, most of the clothes and all the noise.

In [ ]:
X_val, _ = get_arrays(mnist_val_loader)
threshold = ...   # TODO: 99th percentile of the reconstruction errors of the validation images
print(f"threshold: {threshold:.4f}")

for name, values in errors.items():
    flagged = ...   # TODO: fraction of images with an error larger than the threshold
    print(f"{name:<24} - flagged as anomalies: {flagged:.1%}")

And which **digits** look the most unusual to the autoencoder? Let's look at the MNIST test images with the largest reconstruction error.

### 🛠 Exercise 18 — The most unusual digits

Find the indices of the 10 MNIST test images with the **largest** reconstruction error. *Hint:* `argsort()` returns the indices that sort an array in **increasing** order: reverse them with `[::-1]` and take the first 10.

In [ ]:
worst = ...   # TODO: indices of the 10 largest errors
plot_images(np.concatenate([X_test[worst], ae_reconstruct(ae_10, X_test[worst])]), n_cols=10,
            titles=[f"{y_test[i]} (error {errors['MNIST (digits)'][i]:.3f})" for i in worst] + [""] * 10,
            suptitle="The most unusual MNIST test digits - top: input, label (error), bottom: reconstruction")

These are badly written, very thick or strangely cut digits: a quick way to find the "weird" examples in a dataset, for instance to check for errors in the data.

**Caveat.** The reconstruction error is a simple and popular anomaly score, but it is not perfect: an autoencoder can sometimes rebuild well also simple inputs that it has never seen (e.g. an almost black image). It is a useful tool, not a guarantee.

## 4.3 Classification with few labels: pre-training and fine-tuning

Now the scenario that makes unsupervised learning so important in practice. Imagine that we have the 48k training images, but we can afford to label **only 10 images per digit** (100 labels in total).

We compare two classifiers with **exactly the same architecture**: the encoder of the autoencoder (`784 → 512 → 256 → 10`) followed by a `Linear(10, 10)` layer that outputs the logits of the 10 classes. The only difference is how the encoder starts:

| | From scratch | Pre-trained |
|---|---|---|
| Encoder at the start | random weights (as in Lab 1) | the weights of the trained autoencoder |
| Data used before | none | the 48k images, **without labels** |
| Training | the 100 labelled images | the same 100 labelled images (**fine-tuning**) |

This is the same recipe used by modern large models: **pre-train** on a lot of unlabelled data with a self-supervised task, then **fine-tune** on the few labelled examples of the task we care about.

### 🛠 Exercise 19 — A classifier on top of an encoder

Complete the `Classifier`:
- in `__init__`, the `head` is a `Linear` layer from the code (`latent_dim` numbers) to the `n_classes` logits;
- in `forward`, flatten the images, apply the encoder to get the codes `z`, and apply the head to get the logits.

`train_one_epoch` and `accuracy` are the same as in Lab 1, `few_labels_subset` picks `n_per_class` labelled images per digit and `train_classifier` puts everything together: read them, but they are complete.

In [ ]:
class Classifier(torch.nn.Module):
    """An encoder (784 -> latent_dim) followed by a linear layer that outputs the logits of the classes."""
    def __init__(self, encoder, latent_dim: int, n_classes: int = 10):
        super().__init__()
        self.encoder = encoder
        self.head = ...   # TODO: latent_dim -> n_classes

    def forward(self, x):
        z = ...           # TODO: (batch, 1, 28, 28) -> (batch, latent_dim)
        return ...        # TODO: (batch, latent_dim) -> (batch, 10) logits


def train_one_epoch(model, dataloader, loss_fn, optimizer):
    """Same as Lab 1: train for one epoch (with the labels!) and return the average loss."""
    model.train()
    epoch_loss = 0
    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        l = loss_fn(model(x), y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        epoch_loss += l.item()
    return epoch_loss / len(dataloader)


def accuracy(model, dataloader):
    """Same as `evaluate` of Lab 1, but returns only the accuracy."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in dataloader:
            correct += (model(x.to(DEVICE)).argmax(dim=1).cpu() == y).sum().item()
    return correct / len(dataloader.dataset)


def few_labels_subset(dataset, n_per_class: int, seed: int = 0):
    """Pick `n_per_class` random images of every digit from `dataset` (a Subset of mnist_train)."""
    rng = np.random.default_rng(seed)
    labels = mnist_train.targets[dataset.indices].numpy()   # labels of the images of the subset
    chosen = [rng.choice(np.where(labels == digit)[0], n_per_class, replace=False) for digit in range(10)]
    return Subset(dataset, np.concatenate(chosen).tolist())


def train_classifier(encoder, labelled_set, epochs: int = 100):
    """Train an encoder + linear head on `labelled_set` and return the test accuracy."""
    torch.manual_seed(0)
    model = Classifier(encoder, latent_dim=LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(labelled_set, batch_size=32, shuffle=True)
    for epoch in range(epochs):
        train_one_epoch(model, loader, torch.nn.CrossEntropyLoss(), optimizer)
    return accuracy(model, mnist_test_loader)

For a fair comparison, both models are trained with the same optimizer, the same number of epochs and the same labelled images. The pre-trained encoder is a **copy** (`copy.deepcopy`) of the one of `ae_10`, so that fine-tuning does not change our autoencoder.

We repeat the experiment with 10 and 100 labelled images per digit. With so few labels the result depends a lot on *which* images we label, so for each case we repeat the experiment with 3 random choices of the labelled images and we report the mean and the standard deviation. (This takes a minute or two.)

### 🛠 Exercise 20 — From scratch vs pre-trained

For every number of labels and every seed, create two encoders with the **same architecture**:
- `scratch_encoder`, with random weights: the `.encoder` of a **new** `AutoEncoder(latent_dim=LATENT_DIM)`;
- `pretrained_encoder`, with the weights learned without labels: a copy of `ae_10.encoder` made with `copy.deepcopy`. Why a copy? Fine-tuning changes the weights: without the copy it would modify `ae_10` itself, and the next seed would start from an encoder that was already fine-tuned!

Then train a classifier on top of each encoder with `train_classifier` and append the test accuracies to the two lists.

In [ ]:
for n_per_class in [10, 100]:
    acc_scratch, acc_pretrained = [], []
    for seed in range(3):   # 3 different random choices of the labelled images
        labelled_set = few_labels_subset(mnist_train_split, n_per_class, seed=seed)
        scratch_encoder = ...      # TODO: random weights
        pretrained_encoder = ...   # TODO: weights learned without labels (a copy!)
        # TODO: train a classifier on each encoder and append the test accuracies
        ...
    print(f"{10 * n_per_class:>5} labelled images - test accuracy "
          f"from scratch: {np.mean(acc_scratch):.3f} ± {np.std(acc_scratch):.3f} - "
          f"pre-trained: {np.mean(acc_pretrained):.3f} ± {np.std(acc_pretrained):.3f}")

With only 100 labels, starting from the pre-trained encoder gives a clearly better classifier than starting from random weights (in our runs, typically 5 to 10 points of accuracy more): the encoder already "knows" how digits are made, the labels are only needed to give names to what it found. With 1000 labels the gap gets smaller, because the model can learn more and more from the labels alone (with all the 48k labels, we saw in Lab 1 that an MLP from scratch reaches ~97%).

**Take-home message:** unsupervised (self-supervised) learning is most useful when **labels are scarce and unlabelled data is abundant**, which is the most common situation in the real world.

# Summary

| | PCA | K-means | Autoencoder |
|---|---|---|---|
| Task | dimensionality reduction | clustering | dimensionality reduction / representation learning |
| Uses labels | no | no | no |
| What it learns | the directions of maximum variance (linear) | $K$ centroids | a non-linear encoder and decoder |
| Loss | reconstruction error (MSE) | distance of each point from its centroid | reconstruction error (MSE) |
| Pros | fast, simple, a single solution | simple, easy to interpret | much better compression, codes that group similar images |
| Cons | linear: limited compression | depends on the distance: poor on raw pixels | needs training, the latent space is not "organised" |

The key points:
- **unsupervised learning finds structure in data without labels**: compressed descriptions, groups, unusual points;
- **the representation matters**: the same K-means works much better on learned codes than on raw pixels;
- an **autoencoder** learns a representation by solving a self-supervised task, *rebuild your own input through a bottleneck*;
- this representation can be reused: for **clustering**, for **anomaly detection** (reconstruction error) and as **pre-training** for a classifier when labels are few.

# Homework

## 1. The size of the bottleneck

Train the `AutoEncoder` with `latent_dim` in `[2, 5, 10, 20, 50]`:
1. Plot the test reconstruction error of the AE and of PCA (same number of components) as a function of `latent_dim`.
2. For every `latent_dim`, compute the clustering accuracy of K-means on the codes.
3. To visualise a latent space with more than 2 dimensions, project the codes of the test set in 2D with `sklearn.manifold.TSNE` (use a subset of ~3000 images, it is slow) and colour them by label. Do the same with the raw pixels.

Questions:
- Does the reconstruction always improve with a larger `latent_dim`? And the clustering?
- When does the gap between AE and PCA get smaller?

## 2. A linear autoencoder is PCA

Remove all the ReLUs and the sigmoid from the `AutoEncoder` (only `Linear` layers) and train it with `latent_dim=2`.

Questions:
- Compare its test reconstruction error with the one of PCA with 2 components. What do you notice?
- Plot its latent space next to the one of PCA: are they the same? (Hint: they span the same plane, but the axes may be rotated and scaled.)